In [15]:
import pickle  
import os      
import numpy as np  
from datetime import datetime, timedelta  
import pandas as pd  
from tqdm import tqdm  
import logging  
import time  
from datetime import timedelta
from sklearn.metrics.pairwise import cosine_similarity

DATA_PATH = "../DG_data/bluesky"
PROCESSED_DATA_PATH = "../processed_data/bluesky"

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
output_file = os.path.join(os.path.expanduser("~"), 'post_dynamic_embeddings.pkl')

# Load the saved embeddings to verify
logger.info("Loading embeddings to verify...")
with open(output_file, 'rb') as f:
    loaded_embeddings = pickle.load(f)
logger.info(f"Successfully loaded {len(loaded_embeddings)} post embeddings")

INFO:__main__:Loading embeddings to verify...
INFO:__main__:Successfully loaded 20944583 post embeddings


In [16]:
post_embeddings_df = pd.DataFrame(loaded_embeddings)

In [17]:
# Load mappings
with open(os.path.join(DATA_PATH, 'post_mapping.pkl'), 'rb') as f:
    post_mapping = pickle.load(f)
with open(os.path.join(DATA_PATH, 'user_mapping.pkl'), 'rb') as f:
    user_mapping = pickle.load(f)

# Create inverse mappings
inv_user_mapping = {idx: did for did, idx in user_mapping.items()}
inv_post_mapping = {idx: uri for uri, idx in post_mapping.items()}

def uri_to_bsky_link(uri):
    """Convert AT URI to bsky.app link"""
    parts = uri.split('/')
    if len(parts) >= 3:
        did = parts[2]
        tid = parts[-1]
        return f"https://bsky.app/profile/{did}/post/{tid}"
    return uri

def did_to_bsky_link(did_uri):
    """Convert DID URI to bsky.app profile link"""
    return f"https://bsky.app/profile/{did_uri}"

def post_did_to_bsky_link(post_did):
    """Convert post DID to bsky.app post link"""
    parts = post_did.split('_')
    if len(parts) == 2:
        profile_did = parts[0]
        post_id = parts[1]
        return f"https://bsky.app/profile/{profile_did}/post/{post_id}"
    return post_did

In [18]:
def get_candidates(user_id, timestamp, post_embeddings_df, user_dynamic_features_df, df, n_candidates=10):
    """
    Get candidate posts for a user at a specific time
    
    Args:
        user_id: ID of the user
        timestamp: Timestamp to get recommendations for
        post_embeddings_df: DataFrame containing post embeddings
        user_dynamic_features_df: DataFrame of user embeddings
        df: Original interactions DataFrame (for filtering)
        n_candidates: Number of candidates to return
    """
    # Get user embedding
    embedding_date = pd.Timestamp(timestamp.date()) + pd.Timedelta(hours=7)
    user_embedding = user_dynamic_features_df.loc[embedding_date, user_id]
    
    if not isinstance(user_embedding, np.ndarray):
        return []
    
    # Get all posts this user has interacted with before this timestamp
    user_previous_posts = set(
        df[
            (df['source_node'] == user_id) & 
            (df['timestamp'] < timestamp)
        ]['destination_node']
    )
    
    # Get posts that were active at this time
    # time_window_start = timestamp - timedelta(hours=24)
    time_window_start = timestamp - timedelta(minutes=20)
    active_posts = post_embeddings_df[
        (post_embeddings_df['timestamp'] <= timestamp) & 
        (post_embeddings_df['timestamp'] >= time_window_start)
    ]
    
    if len(active_posts) == 0:
        return []
    
    # Get latest embedding for each post and filter out previously interacted posts
    latest_embeddings = (
        active_posts.groupby('post_id')
        .last()
        .reset_index()
    )
    latest_embeddings = latest_embeddings[
        ~latest_embeddings['post_id'].isin(user_previous_posts)
    ]
    
    if len(latest_embeddings) == 0:
        return []
    
    # Calculate similarities
    post_embeddings = np.stack(latest_embeddings['embedding'].values)
    similarities = cosine_similarity([user_embedding], post_embeddings)[0]
    
    # Get top N candidates
    top_indices = np.argsort(similarities)[-n_candidates:][::-1]
    candidates = latest_embeddings.iloc[top_indices]
    
    return list(zip(candidates['post_id'], similarities[top_indices]))

In [28]:
logger.info("Loading data...")
df = pd.read_csv(os.path.join(DATA_PATH, 'bluesky.csv'))
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')

# Load user dynamic features from pickle file
with open(os.path.join(DATA_PATH, 'user_dynamic_features.pkl'), 'rb') as f:
    user_dynamic_features = pickle.load(f)

# Convert user features dictionary to DataFrame for easier manipulation
user_dynamic_features_df = pd.DataFrame.from_dict(user_dynamic_features, orient='index')
user_dynamic_features_df.index = pd.to_datetime(user_dynamic_features_df.index, unit='s')
user_dynamic_features_df = user_dynamic_features_df.sort_index()

# Add embedding date column (7am of each day) for temporal alignment
df['embedding_date'] = df['timestamp'].dt.date.apply(
    lambda x: pd.Timestamp(x) + pd.Timedelta(hours=7)
)

INFO:__main__:Loading data...


In [29]:
# Get a random interaction
random_interaction = df.sample(1).iloc[0]
user_id = random_interaction['source_node']
timestamp = random_interaction['timestamp']

# Get human readable IDs
user_did = inv_user_mapping.get(user_id, f"Unknown user {user_id}")
post_did = inv_post_mapping.get(random_interaction['destination_node'], "Unknown post")

print(f"Selected random interaction:")
print(f"User: {did_to_bsky_link(user_did)}")
print(f"Time: {timestamp}")
print(f"Post interacted with: {post_did_to_bsky_link(post_did)}")

# Get recommendations
recommendation_time = timestamp - pd.Timedelta(seconds=1)

candidates = get_candidates(
    user_id, 
    recommendation_time,
    post_embeddings_df, 
    user_dynamic_features_df,
    df
)

# Get future interactions for this user (within next 24 hours)
future_interactions = set(
    df[
        (df['source_node'] == user_id) & 
        (df['timestamp'] > timestamp) &
        (df['timestamp'] <= timestamp + pd.Timedelta(hours=24))
    ]['destination_node']
)

print(f"\nTop candidates for user {did_to_bsky_link(user_did)} at {recommendation_time}:")
for post_id, similarity in candidates:
    post_did = inv_post_mapping.get(post_id, f"Unknown post {post_id}")
    post_link = post_did_to_bsky_link(post_did)
    
    # Check if user interacted with this post later
    future_interaction = "✅" if post_id in future_interactions else " "
    
    print(f"{future_interaction} Post: {post_link} (similarity = {similarity:.3f})")

# Print some stats
n_hits = sum(1 for post_id, _ in candidates if post_id in future_interactions)
if n_hits > 0:
    print(f"\n✨ The user later interacted with {n_hits} of our recommended posts!")
else:
    print(f"\nThe user didn't interact with any of our recommended posts in the next 24 hours.")

Selected random interaction:
User: https://bsky.app/profile/did:plc:nejscld5zlavytijxz27qhvj
Time: 2023-05-03 12:38:07
Post interacted with: https://bsky.app/profile/did:plc:3la4hhaon456xbzr3q6yd7i4/post/3jusb6paz7s2v

Top candidates for user https://bsky.app/profile/did:plc:nejscld5zlavytijxz27qhvj at 2023-05-03 12:38:06:
  Post: https://bsky.app/profile/did:plc:czze3j5772nu6gxdhben5i34/post/3jut6zjlsmb27 (similarity = 0.785)
  Post: https://bsky.app/profile/did:plc:szxy7csg2qux6pqyihbdsoja/post/3jut6ml3hor2b (similarity = 0.785)
  Post: https://bsky.app/profile/did:plc:czze3j5772nu6gxdhben5i34/post/3jut77hftsr2b (similarity = 0.785)
  Post: https://bsky.app/profile/did:plc:64ryvurqwzr6ljn5v7lwninh/post/3jut6e35qu72e (similarity = 0.780)
  Post: https://bsky.app/profile/did:plc:5ggrdjcno27omroqpqljpqqp/post/3jut6w7e4my2b (similarity = 0.756)
  Post: https://bsky.app/profile/did:plc:gttrfs4hfmrclyxvwkwcgpj7/post/3jut6vrs3w42m (similarity = 0.722)
  Post: https://bsky.app/profile/did:pl

In [30]:
df

,source_node,destination_node,timestamp,edge_label,embedding_date
0,12248,1349,2023-01-01 02:43:21,0,2023-01-01 07:00:00
1,50947,3044497,2023-01-01 02:49:54,0,2023-01-01 07:00:00
2,24218,2347863,2023-01-01 03:52:02,0,2023-01-01 07:00:00
3,13743,1349,2023-01-01 05:16:55,0,2023-01-01 07:00:00
4,50947,1349,2023-01-01 05:35:02,0,2023-01-01 07:00:00
...,...,...,...,...,...
22131393,94142,380415,2023-06-30 23:59:58,0,2023-06-30 07:00:00
22131394,103308,47287,2023-06-30 23:59:59,0,2023-06-30 07:00:00
22131395,87720,1073032,2023-06-30 23:59:59,0,2023-06-30 07:00:00
22131396,27780,1077586,2023-06-30 23:59:59,0,2023-06-30 07:00:00
